# 067 — Reconocimiento automático del habla

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Señal → features:** audio muestreado a 16 kHz → tramas de 25 ms (hop 10 ms) → FFT →
filtros en escala **mel** + log → log-mel espectrograma (entrada de Whisper, 80 bandas).
Los **MFCC** (DCT sobre el log-mel) fueron el estándar de la era HMM.

**Alineación:** ~300 tramas para ~10 palabras. **CTC** entrena sin alineación explícita:
emite símbolo o blanco `∅` por trama y suma la probabilidad de todas las alineaciones que
colapsan a la transcripción. **Whisper** (2022) usa en cambio un transformer
encoder-decoder que genera texto token a token, entrenado con 680 000 h de supervisión
débil: robusto sin fine-tuning, pero autoregresivo (lento) y capaz de alucinar texto en
silencios.

**Métrica:** `WER = (S+D+I)/N` con alineación de Levenshtein por palabra; puede superar
100 % y debe desglosarse por subgrupo de hablantes (acentos, edad, género).


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Ejercicios

**Ejercicio 1 — WER a mano.** Referencia: `activa la alarma de las siete` (6 palabras).
Hipótesis: `activa la alarma a las siete y media`. Alinea, cuenta S/D/I y calcula el WER.
¿Puede el WER superar el 100 %? Da un ejemplo mínimo.

**Ejercicio 2 — Tramas y dimensiones.** Un audio de 5 s a 16 kHz se procesa con ventanas
de 25 ms y hop de 10 ms, 80 bandas mel. Calcula cuántas muestras tiene la señal, cuántas
tramas produce (aprox.) y la forma de la matriz de entrada al modelo.

**Ejercicio 3 — CTC en voz.** La red emite por trama: `∅ h ∅ o o ∅ l l l ∅ a a`.
Colapsa la salida. ¿Qué transcribiría sin el símbolo blanco y por qué el blanco es aún más
necesario en voz que en OCR (piensa en la duración de los sonidos)?

**Ejercicio 4 — WER en código.** Implementa el WER con distancia de Levenshtein sobre
listas de palabras y verifica el ejercicio 1.


In [ ]:
# TODO: ejecuta run_lab("perception", seed=67)
# TODO: comprueba que el resultado incluya las claves 'kind' y 'evidence'
result = None


In [ ]:
# TU RESPUESTA AQUÍ
# Ejercicio 4: WER con Levenshtein sobre palabras
def wer(ref, hyp):
    r, h = ref.split(), hyp.split()
    # completa: matriz de programación dinámica (como CER, pero sobre palabras)
    return None

ref = "activa la alarma de las siete"
hyp = "activa la alarma a las siete y media"
# imprime wer(ref, hyp)


## Reflexión

1. Tu ASR reporta WER 8 % global, pero el subtitulado falla sistemáticamente con hablantes
   andinos. ¿Qué desglose de evaluación faltó y qué datos corregirían el problema?
2. ¿Por qué la arquitectura autoregresiva de Whisper lo hace propenso a "transcribir"
   música o silencio, y qué componente previo del pipeline lo mitiga?
3. En una consulta médica transcrita, un WER de 5 % ¿es aceptable? ¿Qué palabras te
   preocupan más que el promedio y cómo las vigilarías?
